In [48]:
import sage.all as sage
import numpy as np
import itertools
from sage.rings.rational_field import QQ
from sage.combinat.sf.sf import SymmetricFunctions


_Sym = SymmetricFunctions(QQ)
_s   = _Sym.schur()
_p   = _Sym.powersum()

Get all $\pi_\lambda \in S_k$ 

In [49]:
def conjugacy_classes_and_character(k: int, lam: list[int]) -> list[tuple]:
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)  # normalize to non-increasing partition

    Sk = sage.SymmetricGroup(k)
    ct = Sk.character_table()
    classes = Sk.conjugacy_classes()

    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    return [(cl.list(), ct[row_idx][j]) for j, cl in enumerate(classes)]

# Should match known character table for S_3
for lam in sage.Partitions(3).list():
    result = conjugacy_classes_and_character(3, lam)
    print(f"\nlambda = {lam}:")
    for elements, chi in result:
        print(f"  sigma = {elements[0]}, chi = {chi}")


lambda = [3]:
  sigma = (), chi = 1
  sigma = (2,3), chi = 1
  sigma = (1,2,3), chi = 1

lambda = [2, 1]:
  sigma = (), chi = 2
  sigma = (2,3), chi = 0
  sigma = (1,2,3), chi = -1

lambda = [1, 1, 1]:
  sigma = (), chi = 1
  sigma = (2,3), chi = -1
  sigma = (1,2,3), chi = 1


In [50]:
from itertools import product as iproduct
import numpy as np

def sigma_matrix_on_tensor_product(sigma, n, k):
    """
    Matrix of sigma in S_k acting on V^{otimes k}, dim(V) = n.
    sigma: element of Sage's SymmetricGroup(k)
    """
    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}
    
    sigma_inv = sigma.inverse()
    dim = n**k
    M = np.zeros((dim, dim), dtype=complex)
    
    for col_idx, basis_vec in enumerate(basis):
        # permute indices by sigma_inv
        new_basis = tuple(basis_vec[sigma_inv(j + 1) - 1] for j in range(k))
        row_idx = index[new_basis]
        M[row_idx, col_idx] = 1.0
    
    return M

In [51]:
def isotypic_projector(lam: list[int], n: int, k: int) -> np.ndarray:
    """
    Compute Pi_lambda on V^{otimes k} where dim(V) = n.
    Uses the character table for efficiency.
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)

    Sk = sage.SymmetricGroup(k)
    ct = Sk.character_table()
    classes = Sk.conjugacy_classes()

    # Match lam to character table row
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    # dimension d_lambda = chi_lambda(identity)
    id_idx = next(j for j, cl in enumerate(classes)
                  if cl.representative().is_one())
    d_lam = int(ct[row_idx][id_idx])

    dim = n**k
    Pi = np.zeros((dim, dim), dtype=complex)

    for j, cl in enumerate(classes):
        chi = complex(ct[row_idx][j])
        # all elements in same class share the same character
        # so compute matrix per element, multiply by chi
        for sigma in cl:
            M = sigma_matrix_on_tensor_product(sigma, n, k)
            Pi += chi * M

    Pi *= d_lam / sage.factorial(k)
    return Pi

In [52]:
n, k = 2, 3  # qubit example: V = C^2, k=3 tensor factors

for lam in sage.Partitions(k).list():
    Pi = isotypic_projector(lam, n, k)
    
    # Check: Pi^2 = Pi (projector)
    assert np.allclose(Pi @ Pi, Pi), f"{lam}: not idempotent"
    
    # Check: Pi is Hermitian
    assert np.allclose(Pi, Pi.conj().T), f"{lam}: not Hermitian"
    
    # Check: trace = d_lambda * multiplicity
    print(f"lambda={lam}, trace={np.trace(Pi).real:.1f}")

# Check: projectors sum to identity
total = sum(isotypic_projector(lam, n, k) 
            for lam in sage.Partitions(k).list())
assert np.allclose(total, np.eye(n**k))
print("Projectors sum to identity ✓")

lambda=[3], trace=4.0
lambda=[2, 1], trace=4.0
lambda=[1, 1, 1], trace=0.0
Projectors sum to identity ✓


In [53]:
import numpy as np
from itertools import product as iproduct

def ket(indices: list[int], n: int) -> np.ndarray:
    """
    Convert a basis vector in ket notation to a vector in (C^n)^{otimes k}.
    
    indices: tuple of ints, e.g. (2, 0, 3) for |2,0,3>
    n: local dimension, e.g. n=4 for C^4
    
    Example: ket((2,0,3), n=4) -> unit vector in C^{4^3} = C^64
    """
    k = len(indices)
    assert all(0 <= i < n for i in indices), f"All indices must be in range [0, {n-1}]"

    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}

    v = np.zeros(n**k, dtype=complex)
    v[index[tuple(indices)]] = 1.0
    return v

# |2,0,3> in (C^4)^{otimes 3}
v = ket([2,0,3], n=4)
print(v.shape)   # (64,)
print(v.sum())   # 1.0 — exactly one nonzero entry

# Recover the index
print(np.argmax(v))  # 2*16 + 0*4 + 3 = 35

(64,)
(1+0j)
35


In [54]:
v = ket([0, 0, 3], n=4)
Pi = isotypic_projector([3], n=4, k=3)

projected = Pi @ v

# Norm of the projected vector — how much of |2,0,3> lives in this subspace
print(np.linalg.norm(projected))

# Applying the projector twice should give the same result
print(np.allclose(Pi @ projected, projected))  # True

# Projectors onto different sectors are orthogonal
Pi_sym  = isotypic_projector([3],     n=4, k=3)
Pi_mix  = isotypic_projector([2,1],   n=4, k=3)
Pi_anti = isotypic_projector([1,1,1], n=4, k=3)

v_sym  = Pi_sym  @ v
v_mix  = Pi_mix  @ v
v_anti = Pi_anti @ v

print(v_anti)

# These three components are orthogonal and reconstruct v
print(np.allclose(v_sym + v_mix + v_anti, v))           # True
print(np.allclose(np.dot(v_sym.conj(), v_mix), 0))      # True

0.5773502691896257
True
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
True
True


In [71]:
def nu(v: list[int], n:int) -> list[int]:
    ret = [0] * n
    for i in v:
        ret[i] += 1
    ret.sort(reverse=True)
    return [i for i in ret if i != 0]

def maps_to_zero(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """
    Check if the isotypic projector Pi_lambda maps v to zero.
    
    lam: Young frame, e.g. [3] or [2,1]
    n:   local dimension
    v:   vector in (C^n)^{otimes k}
    tol: numerical tolerance
    """


    k = sum(lam)
    ketv = ket(v, n)
    Pi = isotypic_projector(lam, n, k)
    projected = Pi @ ketv

    print(f"lam = {str(lam)}, v={str(v)}, n={n}, h(lam)={len(lam)}, h(nu)={len(nu(v,n))}, expected={"0" if len(nu(v,n)) < len(lam) else "any"}, res={"0" if np.linalg.norm(projected) < tol else "|*>"}")

    return bool(np.linalg.norm(projected) < tol)

In [72]:
n = 2
v = [0, 0, 1] # some vector in (C^2)^{otimes 3}

print(maps_to_zero([3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([1,1,1], n=n, v=v))  # True  — no antisymmetric component (n<k)
print(maps_to_zero([2,1],   n=n, v=v))  # depends on the vector

lam = [3], v=[0, 0, 1], n=2, h(lam)=1, h(nu)=2, expected=any, res=|*>
False
lam = [1, 1, 1], v=[0, 0, 1], n=2, h(lam)=3, h(nu)=2, expected=0, res=0
True
lam = [2, 1], v=[0, 0, 1], n=2, h(lam)=2, h(nu)=2, expected=any, res=|*>
False


In [57]:
n = 3
v = [0, 0, 0, 1, 1, 2]

print(maps_to_zero([6],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([3,3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero([2,2,1,1], n=n, v=v))
print(maps_to_zero([2,1,1,1,1], n=n, v=v))
print(maps_to_zero([3,2,1], n=n, v=v))
print(maps_to_zero([4,2],   n=n, v=v))  # depends on the vector

lam = [6], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=1, h(nu)=3, expected=any, res=|*>
False
lam = [3, 3], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=2, h(nu)=3, expected=any, res=|*>
False
lam = [2, 2, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=4, h(nu)=3, expected=0, res=0
True
lam = [2, 1, 1, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=5, h(nu)=3, expected=0, res=0
True
lam = [3, 2, 1], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=3, h(nu)=3, expected=any, res=|*>
False
lam = [4, 2], v=[0, 0, 0, 1, 1, 2], n=3, h(lam)=2, h(nu)=3, expected=any, res=|*>
False


In [ ]:
def v_cumulative(v: list[int]):
    c = v.copy()
    for i in range(1,len(c)):
        c[i] += c[i-1]
    return c


def majorizes(nuv: list[int], lam: list[int]) -> bool:
    assert sum(lam) == sum(nuv)
    lams = 0
    vs = 0
    for i in range(min(len(lam), len(nuv))):
        lams += lam[i]
        vs += nuv[i]
        print(lams, vs)
        if vs > lams:
            return True
    return False

def maps_to_zero_major(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """
    Check if the isotypic projector Pi_lambda maps v to zero.
    
    lam: Young frame, e.g. [3] or [2,1]
    n:   local dimension
    v:   vector in (C^n)^{otimes k}
    tol: numerical tolerance
    """


    k = sum(lam)
    ketv = ket(v, n)
    Pi = isotypic_projector(lam, n, k)
    projected = Pi @ ketv

    print(f"lam = {str(lam)}, v={str(v)}, n={n}, lam_c={v_cumulative(lam)}, nu_c={v_cumulative(nu(v,n))}, expected={"0" if majorizes( nu(v,n),lam) else "any"}, res={"0" if np.linalg.norm(projected) < tol else "|*>"}")

    return bool(np.linalg.norm(projected) < tol)


In [76]:
n = 3
v = [0, 0, 0, 1, 1, 2]

print(maps_to_zero_major([6],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero_major([3,3],     n=n, v=v))  # False — has symmetric component
print(maps_to_zero_major([2,2,1,1], n=n, v=v))
print(maps_to_zero_major([2,1,1,1,1], n=n, v=v))
print(maps_to_zero_major([3,2,1], n=n, v=v))
print(maps_to_zero_major([4,2],   n=n, v=v))  # depends on the vector

6 0
lam = [6], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[6], nu_c=[3, 5, 6], expected=any, res=|*>
False
3 0
lam = [3, 3], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[3, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False
2 0
lam = [2, 2, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[2, 4, 5, 6], nu_c=[3, 5, 6], expected=any, res=0
True
2 0
lam = [2, 1, 1, 1, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[2, 3, 4, 5, 6], nu_c=[3, 5, 6], expected=any, res=0
True
3 0
lam = [3, 2, 1], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[3, 5, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False
4 0
lam = [4, 2], v=[0, 0, 0, 1, 1, 2], n=3, lam_c=[4, 6], nu_c=[3, 5, 6], expected=any, res=|*>
False
